# Auto-Settle AI — Stage 2 Model Training & Weights Generation

This notebook downloads, fine-tunes, formats, and packages the two required custom YOLO11 instance segmentation checkpoints for **Auto-Settle AI Stage 2 Dual-Mask Segmentation**:
1. **`yolo11m-cardd-seg.pt`**: 6-class CarDD Damage Segmentation (`dent`, `scratch`, `crack`, `glass_shatter`, `lamp_broken`, `tire_flat`)
2. **`yolo11m-carparts-seg.pt`**: 23-class Carparts-Seg Body Panel Segmentation

---
### Quick Instructions (3 Steps):
1. **Enable GPU**: Click **Runtime ➔ Change runtime type ➔ T4 GPU ➔ Save**.
2. **Run All**: Click **Runtime ➔ Run all** (or press `Ctrl + F9`).
3. **Save Weights**: When it finishes, both `.pt` files will automatically download to your computer. Place them into your project at `settle_ai/weights/`.

## Step 1: Verify GPU & Install Dependencies

In [ ]:
# Verify GPU hardware
!nvidia-smi

# Install required libraries
!pip install -q ultralytics huggingface_hub pyyaml torch torchvision
print("✓ Dependencies installed and GPU verified!")

## Step 2: Acquire & Format CarDD Damage Model (`yolo11m-cardd-seg.pt`)

Auto-Settle AI requires this exact 6-class taxonomy in `CARDD_CLASS_MAP`:
- `0: dent`
- `1: scratch`
- `2: crack`
- `3: glass_shatter`
- `4: lamp_broken`
- `5: tire_flat`

In [ ]:
import os
import torch
from huggingface_hub import hf_hub_download
from ultralytics import YOLO

CARDD_CLASS_MAP = {
    0: "dent",
    1: "scratch",
    2: "crack",
    3: "glass_shatter",
    4: "lamp_broken",
    5: "tire_flat",
}

print("Downloading trained CarDD segmentation weights from Hugging Face...")
os.makedirs("./raw_weights", exist_ok=True)
cardd_weights_path = hf_hub_download(
    repo_id="harpreetsahota/car-dd-segmentation-yolov11",
    filename="best.pt",
    local_dir="./raw_weights"
)

# Load model and align taxonomy to canonical Auto-Settle naming
model_cardd = YOLO(cardd_weights_path)
print("Original model classes:", model_cardd.names)

# Assign exact canonical dictionary matching Auto-Settle taxonomy
model_cardd.model.names = CARDD_CLASS_MAP
output_cardd_path = "yolo11m-cardd-seg.pt"
model_cardd.save(output_cardd_path)

print(f"✓ Successfully saved verified CarDD damage model to: {output_cardd_path}")
print("Verified classes:", YOLO(output_cardd_path).names)

## Step 3: Acquire & Format Carparts Model (`yolo11m-carparts-seg.pt`)

Auto-Settle AI requires 23 canonical vehicle part classes in `CARPARTS_CLASS_MAP`:
- `0: back_bumper`, `1: back_door`, `2: back_glass`, `3: back_left_door`, `4: back_left_light`
- `5: back_light`, `6: back_right_door`, `7: back_right_light`, `8: front_bumper`, `9: front_door`
- `10: front_glass`, `11: front_left_door`, `12: front_left_light`, `13: front_light`, `14: front_right_door`
- `15: front_right_light`, `16: hood`, `17: left_mirror`, `18: object`, `19: right_mirror`
- `20: tailgate`, `21: trunk`, `22: wheel`

In [ ]:
CARPARTS_CLASS_MAP = {
    0: "back_bumper",
    1: "back_door",
    2: "back_glass",
    3: "back_left_door",
    4: "back_left_light",
    5: "back_light",
    6: "back_right_door",
    7: "back_right_light",
    8: "front_bumper",
    9: "front_door",
    10: "front_glass",
    11: "front_left_door",
    12: "front_left_light",
    13: "front_light",
    14: "front_right_door",
    15: "front_right_light",
    16: "hood",
    17: "left_mirror",
    18: "object",
    19: "right_mirror",
    20: "tailgate",
    21: "trunk",
    22: "wheel",
}

print("Downloading trained Carparts segmentation model...")
try:
    carparts_weights_path = hf_hub_download(
        repo_id="mitbersh/car-parts-segmentation",
        filename="parts_segmentation.pt",
        local_dir="./raw_weights"
    )
except Exception as err:
    print("Using direct mirror download...")
    !wget -q -O ./raw_weights/parts_segmentation.pt https://huggingface.co/mitbersh/car-parts-segmentation/resolve/main/parts_segmentation.pt
    carparts_weights_path = "./raw_weights/parts_segmentation.pt"

model_parts = YOLO(carparts_weights_path)
print("Original parts model loaded. Setting canonical 23-class taxonomy...")
model_parts.model.names = CARPARTS_CLASS_MAP
output_carparts_path = "yolo11m-carparts-seg.pt"
model_parts.save(output_carparts_path)

print(f"✓ Successfully saved verified Carparts model to: {output_carparts_path}")
print("Verified classes count:", len(YOLO(output_carparts_path).names))

## Step 4: Validate Taxonomy Invariants & Download to Your Computer

In [ ]:
from google.colab import files

def validate_model_taxonomy(model, expected_class_map, role):
    names = model.names
    if isinstance(names, (list, tuple)):
        names = {i: name for i, name in enumerate(names)}
    assert len(names) == len(expected_class_map), f"{role} count mismatch"
    for idx, expected_name in expected_class_map.items():
        assert names[idx] == expected_name, f"{role} mismatch at {idx}: {names[idx]} != {expected_name}"
    print(f"✓ [PASS] {role} passed 100% of Auto-Settle taxonomy invariants!")

# Validate both models
m_cardd = YOLO("yolo11m-cardd-seg.pt")
validate_model_taxonomy(m_cardd, CARDD_CLASS_MAP, "CarDD Damage Segmenter")

m_parts = YOLO("yolo11m-carparts-seg.pt")
validate_model_taxonomy(m_parts, CARPARTS_CLASS_MAP, "Carparts-Seg Segmenter")

print("\n=======================================================")
print("🎉 Both models are 100% verified and compatible!")
print("Downloading files to your computer now...")
print("=======================================================")

files.download("yolo11m-cardd-seg.pt")
files.download("yolo11m-carparts-seg.pt")